# ResNet18 CIFAR-10 Static Quantization with FedCore

This notebook is a notebook-oriented version of the supplied training/quantization script.

Pipeline:

1. Set reproducibility parameters.
2. Load and prepare CIFAR-10.
3. Initialize a pretrained ResNet18 and adapt it to CIFAR-10.
4. Build `CompressionInputData`.
5. Configure FedCore for static quantization.
6. Run FedCore compression.
7. Inspect and save the resulting model comparison metrics.

The notebook keeps the original FedCore configuration and experiment logic, but removes the CLI-specific `argparse`/`main()` wrapper so individual stages can be executed and inspected interactively.

In [ ]:
from __future__ import annotations

import copy
import csv
import json
import logging
import random
import statistics
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from fedot.core.repository.tasks import Task, TaskTypesEnum
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18

from fedcore.algorithm.quantization.quantizers import BaseQuantizer
from fedcore.algorithm.quantization.utils import QDQWrapper, QDQWrapping
from fedcore.data.data import CompressionInputData
from fedcore.models.backbone.convolutional.resnet import CLF_MODELS

from fedcore.api.config_factory import ConfigFactory
from fedcore.api.api_configs import (
    APIConfigTemplate,
    AutoMLConfigTemplate,
    FedotConfigTemplate,
    LearningConfigTemplate,
    ModelArchitectureConfigTemplate,
    TrainingTemplate,
    DeviceConfigTemplate,
    ComputeConfigTemplate,
    QuantizationTemplate,
)
from fedcore.api.main import FedCore


## 1. Repository paths and configuration

In [ ]:
REPO_ROOT = Path.cwd().resolve()
MODEL_EXPORTER_DIR = REPO_ROOT / "model_exporter"

# If the notebook is located in a subdirectory of the repository, adjust this
# to the repository root explicitly when needed.
if not (REPO_ROOT / "fedcore").exists() and (REPO_ROOT.parent / "fedcore").exists():
    REPO_ROOT = REPO_ROOT.parent
    MODEL_EXPORTER_DIR = REPO_ROOT / "model_exporter"

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(MODEL_EXPORTER_DIR))

print("REPO_ROOT:", REPO_ROOT)
print("Python:", sys.version)
print("PyTorch:", torch.__version__)


In [ ]:
@dataclass
class Config:
    """Parameters of the reproducible quantization experiment."""

    seed: int = 42
    epochs: int = 10
    train_samples: int = 50_000
    test_samples: int = 10_000
    calibration_samples: int = 1_024
    batch_size: int = 64
    learning_rate: float = 1e-3
    output_dir: Path = REPO_ROOT / "results" / "quantization_resnet18"
    data_dir: Path = REPO_ROOT / "datasets"


config = Config()
config.output_dir.mkdir(parents=True, exist_ok=True)

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)

print(config)


## 2. CIFAR-10 data

In [ ]:
class Cifar10Data:
    """Load CIFAR-10 and prepare train/test/calibration DataLoaders."""

    def __init__(self, config: Config, one_hot: bool = True):
        self.one_hot = one_hot
        self.num_classes = 10

        transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465),
                    (0.2470, 0.2435, 0.2616),
                ),
            ]
        )

        train = datasets.CIFAR10(
            config.data_dir,
            train=True,
            download=True,
            transform=transform,
        )
        test = datasets.CIFAR10(
            config.data_dir,
            train=False,
            download=True,
            transform=transform,
        )

        generator = torch.Generator().manual_seed(config.seed)
        train_ids = torch.randperm(len(train), generator=generator)
        test_ids = torch.randperm(len(test), generator=generator)

        n_train = min(config.train_samples, len(train))
        n_test = min(config.test_samples, len(test))
        n_calib = min(config.calibration_samples, n_train)

        self.train = DataLoader(
            Subset(train, train_ids[:n_train]),
            batch_size=config.batch_size,
            shuffle=True,
            num_workers=0,
            collate_fn=self._collate_one_hot if one_hot else None,
        )

        self.calibration = DataLoader(
            Subset(train, train_ids[:n_calib]),
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=self._collate_one_hot if one_hot else None,
        )

        self.test = DataLoader(
            Subset(test, test_ids[:n_test]),
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=self._collate_one_hot if one_hot else None,
        )

    def _collate_one_hot(self, batch):
        data = torch.stack([item[0] for item in batch])
        labels = torch.stack(
            [
                torch.nn.functional.one_hot(
                    torch.tensor(item[1]),
                    num_classes=10,
                ).float()
                for item in batch
            ]
        )
        return data, labels


data = Cifar10Data(config)

print(
    f"train={len(data.train.dataset)}, "
    f"test={len(data.test.dataset)}, "
    f"calibration={len(data.calibration.dataset)}"
)


In [ ]:
# Inspect one batch
example_batch = next(iter(data.train))
example_input, example_target = example_batch

print("Input shape:", example_input.shape)
print("Target shape:", example_target.shape)
print("Input dtype:", example_input.dtype)
print("Target dtype:", example_target.dtype)


## 3. Build and adapt ResNet18

In [ ]:
# Load ImageNet-pretrained ResNet18.
model = resnet18(pretrained=True)

# CIFAR-10 has 10 classes.
model.fc = nn.Linear(model.fc.in_features, 10)

# CIFAR-10 images are 32x32, so use a CIFAR-style stem.
model.conv1 = nn.Conv2d(
    3,
    64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False,
)
model.maxpool = nn.Identity()

print(model)


In [ ]:
# Basic model sanity check.
model.eval()

with torch.no_grad():
    logits = model(example_input)

print("Logits shape:", logits.shape)


## 4. Construct `CompressionInputData`

In [ ]:
compression_data = CompressionInputData(
    features=example_input,
    target=model,
    train_dataloader=data.train,
    val_dataloader=data.calibration,
    test_dataloader=data.test,
    task=Task(TaskTypesEnum.classification),
    input_dim=example_input.size(-1),
)

print("CompressionInputData created.")


## 5. Configure FedCore static quantization

In [ ]:
fedot_config = FedotConfigTemplate(
    problem="classification",
    metric=[
        "MulticlassAccuracy__10",
        "Latency",
        "ModelSize",
    ],
    pop_size=1,
    timeout=0.1,
    initial_assumption=model,
)

quantization_config = QuantizationTemplate(
    quant_type="static",
    allow_emb=False,
    allow_conv=True,
)

automl_config = AutoMLConfigTemplate(
    fedot_config=fedot_config,
)

learning_config = LearningConfigTemplate(
    criterion="cross_entropy",
    learning_strategy="from_checkpoint",
    peft_strategy_params=[quantization_config],
)

api_template = APIConfigTemplate(
    automl_config=automl_config,
    learning_config=learning_config,
)

APIConfig = ConfigFactory.from_template(api_template)
api_config = APIConfig()

print("FedCore API configuration created.")


In [ ]:
# Optional: inspect the generated configuration object.
api_config


## 6. Run FedCore quantization

In [ ]:
fedcore_compressor = FedCore(api_config)

start_time = time.perf_counter()
fedcore_compressor.fit(compression_data)
elapsed = time.perf_counter() - start_time

print(f"FedCore compression finished in {elapsed:.2f} s")


## 7. Get and inspect metrics

In [ ]:
model_comparison = fedcore_compressor.get_report(compression_data)

display(model_comparison)


In [ ]:
# Inspect available columns and basic statistics.
print("Columns:", list(model_comparison.columns))
display(model_comparison.describe(include="all"))


## 8. Save results

In [ ]:
save_path = REPO_ROOT / "results" / "quantization_resnet18"
save_path.mkdir(parents=True, exist_ok=True)

metrics_path = save_path / "metrics.csv"
model_comparison.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)


## Notes

- The original script defined `epochs` and `learning_rate`, but the FedCore configuration shown there does not use them directly. They are therefore retained in `Config` for experiment bookkeeping.
- `Cifar10Data` preserves the original one-hot target representation.
- The pretrained ImageNet ResNet18 is adapted after loading; in particular, the replacement `fc` layer and CIFAR-style convolutional stem are newly initialized.
- The notebook uses `Path.cwd()` rather than `Path(__file__)`, because notebooks do not have `__file__`. If the notebook is launched from another working directory, set `REPO_ROOT` explicitly.
